In [1]:
import os
import sys
from pyspark.context import SparkContext, SparkConf

In [2]:
# Java
java_home = r"C:\Users\n.osipov\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

# Принудительно указываем Python из текущего окружения
python_path = sys.executable
print("Используем Python:", python_path)

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setAppName("SkillBox")
        .setMaster("local[1]")                    # один поток — проще для диагностики
        .set("spark.driver.host", "127.0.0.1")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.pyspark.python", python_path)
        .set("spark.pyspark.driver.python", python_path)
       )

sc = SparkContext(conf=conf)
print(sc.uiWebUrl)

Используем Python: C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\python.exe


C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


http://127.0.0.1:4040


In [3]:
rdd = sc.parallelize(range(10))

In [4]:
rdd

PythonRDD[1] at RDD at PythonRDD.scala:59

In [5]:
rdd.take(5)

[0, 1, 2, 3, 4]

In [11]:
rdd.take(20)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [13]:
rdd.map(lambda x: (x, x**2, x**3)).take(5)

[(0, 0, 0), (1, 1, 1), (2, 4, 8), (3, 9, 27), (4, 16, 64)]

In [14]:
rdd.take(5)

[0, 1, 2, 3, 4]

In [16]:
rdd_2 = rdd.map(lambda x: (x, x**2, x**3))

In [18]:
rdd_2.take(10)

[(0, 0, 0),
 (1, 1, 1),
 (2, 4, 8),
 (3, 9, 27),
 (4, 16, 64),
 (5, 25, 125),
 (6, 36, 216),
 (7, 49, 343),
 (8, 64, 512),
 (9, 81, 729)]

In [25]:
rdd_3 = rdd.filter(lambda x: x > 5)

In [26]:
rdd_3.take(10)

[6, 7, 8, 9]

In [28]:
rdd.flatMap(lambda x: (x, x**2, x**3)).take(20)

[0, 0, 0, 1, 1, 1, 2, 4, 8, 3, 9, 27, 4, 16, 64, 5, 25, 125, 6, 36]

In [30]:
collect_rdd = rdd.map(lambda x: (x, x**2, x**3)).collect()

In [31]:
type(collect_rdd)

list

In [32]:
from operator import add
rdd.reduce(add)

45

In [33]:
type(rdd.reduce(add))

int

In [38]:
rdd_reduce = sc.parallelize([("a", 1), ("b", 2), ("a", 3)])

In [39]:
rdd_reduce.reduceByKey(add).take(4)

[('a', 4), ('b', 2)]

In [42]:
rdd_reduce.groupByKey().map(lambda x: (x[0], sum(x[1]))).take(5)

[('a', 4), ('b', 2)]

In [55]:
text = ["Всем привет! Меня зовут Никита, а вас как? Зовут игорь?"]

In [56]:
text = sc.parallelize(text)

In [57]:
text.take(5)

['Всем привет! Меня зовут Никита, а вас как? Зовут игорь?']

In [58]:
words = text.flatMap(lambda x: x.lower().split(" "))

In [59]:
words.take(5)

['всем', 'привет!', 'меня', 'зовут', 'никита,']

In [60]:
words.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x+y).take(10)

[('всем', 1),
 ('привет!', 1),
 ('меня', 1),
 ('зовут', 2),
 ('никита,', 1),
 ('а', 1),
 ('вас', 1),
 ('как?', 1),
 ('игорь?', 1)]